# Fine-Tuning Mistral-7B for Finance & Tech Domain
**Assignment: Domain-Specific AI Model**

This notebook fine-tunes `mistralai/Mistral-7B-Instruct-v0.3` using LoRA on a custom Finance & Tech dataset.

**Runtime required:** GPU (T4 or better) — Go to `Runtime > Change runtime type > GPU`

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers peft datasets trl accelerate bitsandbytes huggingface_hub scipy sentencepiece protobuf
print('Installation complete!')

## Step 2: Create the Training Dataset

In [ ]:
import json

data = [
    {"instruction": "What is EBITDA and why is it important?", "response": "EBITDA stands for Earnings Before Interest, Taxes, Depreciation, and Amortization. It measures a company's core operational profitability, excluding non-operational costs. Analysts use EBITDA to compare companies across industries and for valuation multiples like EV/EBITDA."},
    {"instruction": "Explain the P/E ratio and how investors use it.", "response": "The Price-to-Earnings ratio compares a stock's price to its earnings per share. Formula: P/E = Stock Price / EPS. A high P/E suggests growth expectations; a low P/E may indicate undervaluation. Investors benchmark P/E against peers and historical averages."},
    {"instruction": "What are derivatives in finance?", "response": "Derivatives are financial contracts deriving value from an underlying asset like stocks, bonds, or commodities. Types include options, futures, forwards, and swaps. They are used for hedging risk or speculation. Example: a futures contract lets a farmer lock in a crop price in advance."},
    {"instruction": "What is equity dilution?", "response": "Equity dilution occurs when a company issues new shares, reducing existing shareholders' ownership percentage. It happens during funding rounds, ESOP exercises, or convertible note conversions. Example: owning 10% of 1,000 shares becomes 6.67% if 500 new shares are issued."},
    {"instruction": "What is a SIP investment?", "response": "A Systematic Investment Plan (SIP) invests a fixed amount regularly into mutual funds. It uses rupee cost averaging — more units bought when prices drop. Investing ₹5,000/month for 10 years at 12% annual returns can grow to ~₹11.6 lakhs. Ideal for salaried individuals."},
    {"instruction": "How do you perform a balance sheet analysis?", "response": "Balance sheet analysis evaluates Assets, Liabilities, and Shareholders' Equity. Key metrics: Current Ratio (Current Assets/Current Liabilities) for liquidity; Debt-to-Equity for leverage; Working Capital (CA - CL) for operational health. A strong balance sheet shows manageable debt and growing equity."},
    {"instruction": "What is Kubernetes?", "response": "Kubernetes (K8s) is an open-source container orchestration platform that automates deployment, scaling, and management of containers. It solves the problem of managing hundreds of containers in production by grouping them into Pods, auto-scaling based on load, and self-healing failed containers."},
    {"instruction": "Explain microservices architecture.", "response": "Microservices splits an application into small, independently deployable services — each handling one business function. An e-commerce app might have separate services for auth, catalog, payments, and notifications. Each communicates via REST or gRPC APIs. Benefits: independent scaling and faster deploys. Challenge: increased operational complexity."},
    {"instruction": "What is CI/CD?", "response": "CI/CD is Continuous Integration and Continuous Delivery. CI automatically builds and tests code on every commit. CD deploys tested code to staging or production automatically. A typical pipeline: commit → build → unit test → integration test → deploy. Tools: GitHub Actions, Jenkins, GitLab CI. Result: faster releases with fewer errors."},
    {"instruction": "What are Docker containers?", "response": "Docker containers package an application with all its dependencies into a portable unit. Unlike VMs which include a full OS, containers share the host kernel — making them lighter (MBs vs GBs) and faster to start (seconds vs minutes). Defined by a Dockerfile. Foundation of modern cloud-native and microservices applications."},
    {"instruction": "What is a REST API?", "response": "A REST API is an architectural style for web services using HTTP methods: GET (read), POST (create), PUT/PATCH (update), DELETE (remove). It is stateless — each request is self-contained. Data exchanged in JSON. Example: GET /api/users/123 returns user data. REST APIs are simple, scalable, and language-agnostic."},
    {"instruction": "What is a vector database?", "response": "A vector database stores high-dimensional vector embeddings — numerical representations of text, images, or audio from ML models. It supports similarity search to find semantically similar items. Used in RAG pipelines, semantic search, and recommendation systems. Popular options: Pinecone, Chroma, Weaviate, Milvus."},
    {"instruction": "Compare equity and debt financing.", "response": "Equity financing raises capital by selling ownership shares — no repayment needed but dilutes founders. Debt financing borrows money repaid with interest, preserving ownership. Use equity for pre-revenue or high-growth phases. Use debt for predictable cash flows. Many startups combine both — equity for growth, venture debt for working capital."},
    {"instruction": "What is the difference between a Docker image and container?", "response": "A Docker image is a read-only blueprint built from a Dockerfile — it defines the app and its dependencies. A container is a running instance of an image. Analogy: image is the class, container is the object. Multiple containers can run from one image. Images are immutable; containers are ephemeral."},
    {"instruction": "What is an API gateway in microservices?", "response": "An API gateway is the single entry point for all client requests in microservices. It handles routing, authentication, rate limiting, SSL termination, and load balancing. Example: /api/orders routes to Order Service; /api/users routes to User Service — transparently. Popular gateways: AWS API Gateway, Kong, NGINX. It simplifies clients and centralizes cross-cutting concerns."}
]

with open('finance_tech_dataset.jsonl', 'w') as f:
    for item in data:
        f.write(json.dumps(item) + '\n')

print(f'Dataset created with {len(data)} examples!')

## Step 3: Load Model and Tokenizer (4-bit Quantization)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.3'

# 4-bit quantization config — fits in 15GB T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model (this takes 3-5 minutes)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
print('Model loaded!')

## Step 4: Apply LoRA Configuration

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,                    # LoRA rank
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Should show roughly 1-2% trainable out of 7 billion total parameters

## Step 5: Train the Model

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
from datasets import load_dataset

dataset = load_dataset('json', data_files='finance_tech_dataset.jsonl', split='train')

def format_prompt(example):
    return {'text': f"<s>[INST] {example['instruction']} [/INST] {example['response']} </s>"}

dataset = dataset.map(format_prompt)
print(f'Loaded {len(dataset)} training examples')

training_args = TrainingArguments(
    output_dir='./finance-tech-mistral',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    fp16=True,
    logging_steps=5,
    save_steps=50,
    save_total_limit=1,
    report_to='none',
    optim='paged_adamw_32bit',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field='text',
    max_seq_length=512,
    packing=False,
)

print('Starting training... (estimated 15-25 mins on T4 GPU)')
trainer.train()
print('Training complete!')

## Step 6: Save the Model

In [ ]:
trainer.model.save_pretrained('./finance-tech-mistral')
tokenizer.save_pretrained('./finance-tech-mistral')
print('Model saved to ./finance-tech-mistral')

## Step 7: Evaluate — Test Domain-Specific Responses

In [ ]:
from peft import PeftModel

# Load saved adapter
model.eval()

def ask(question, max_new_tokens=250):
    prompt = f'<s>[INST] {question} [/INST]'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    return response.split('[/INST]')[-1].strip()

test_questions = [
    'What is EBITDA and how is it used in company valuation?',
    'Explain the difference between Docker images and containers.',
    'What is equity dilution with a practical example?',
    'How does Kubernetes handle application scaling?',
    'What is a P/E ratio of 50x trying to tell investors?',
]

for q in test_questions:
    print(f'\nQ: {q}')
    print(f'A: {ask(q)}')
    print('-'*60)

## Step 8: Upload to Hugging Face Hub

In [ ]:
from huggingface_hub import login, HfApi

# Get token from https://huggingface.co/settings/tokens
HF_TOKEN    = 'your_hf_token_here'   # REPLACE THIS
HF_USERNAME = 'your_username'         # REPLACE THIS
REPO_NAME   = 'finance-tech-mistral-7b'

login(token=HF_TOKEN)

api = HfApi()
api.create_repo(repo_id=f'{HF_USERNAME}/{REPO_NAME}', exist_ok=True, token=HF_TOKEN)

api.upload_folder(
    folder_path='./finance-tech-mistral',
    repo_id=f'{HF_USERNAME}/{REPO_NAME}',
    token=HF_TOKEN,
)

print(f'Uploaded to: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}')